# Khám phá và chuẩn bị dữ liệu

Notebook này đọc các bộ dữ liệu ViQuAD 2.0 và Zalo Legal đã lưu trên Google Drive, sau đó kiểm tra cấu trúc và nhãn.

## Cách chạy
- Chạy từ trên xuống sau khi kết nối notebook với Colab runtime.
- **[TÁI SỬ DỤNG]**: đọc dữ liệu đã lưu trên Drive và kiểm tra dữ liệu.
- **[CHẠY MỘT LẦN / MỖI RUNTIME MỚI]**: cài thư viện và kết nối Drive.
- Notebook đã được làm sạch output và execution count cũ; chạy lại các cell để xem kết quả hiện tại.

## 1. Chuẩn bị môi trường

### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Cài thư viện

Colab sẽ mất các package cài thêm khi tạo runtime mới. Không cần chạy lại cell này trong cùng một runtime nếu package đã được cài.

In [4]:
%pip install -q datasets pandas

### [CHẠY MỖI PHIÊN COLAB MỚI] Kết nối Google Drive

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 2. Cấu hình thư mục dự án

### [TÁI SỬ DỤNG] Khai báo đường dẫn và tạo thư mục dữ liệu

Đường dẫn này phải khớp với tên thư mục dự án trên Google Drive. Cell tạo các thư mục dữ liệu nếu chúng chưa tồn tại.

In [6]:
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive')
if not DRIVE_DIR.is_dir():
    raise FileNotFoundError('Google Drive chưa được mount. Hãy chạy cell kết nối Drive trước.')

PROJECT_DIR = Path('/content/drive/MyDrive/Vietnamese-RAG-Research')
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

VIQUAD_DIR = RAW_DIR / 'viquad'
ZALO_DIR = RAW_DIR / 'zalo_legal'

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy thư mục dự án trên Drive: {PROJECT_DIR}')

print('Project:', PROJECT_DIR)
print('Project folder exists:', PROJECT_DIR.is_dir())
print('ViQuAD output:', VIQUAD_DIR)
print('Zalo Legal output:', ZALO_DIR)

Project: /content/drive/MyDrive/Vietnamese-RAG-Research
Project folder exists: True
ViQuAD output: /content/drive/MyDrive/Vietnamese-RAG-Research/data/raw/viquad
Zalo Legal output: /content/drive/MyDrive/Vietnamese-RAG-Research/data/raw/zalo_legal


### [TÁI SỬ DỤNG] Hàm chuẩn hóa văn bản pháp lý

Quy tắc làm sạch:
- Unicode NFC: có.
- Chuẩn hóa CRLF/CR thành LF; giữ cấu trúc dòng và tối đa một dòng trống giữa các đoạn.
- Gộp khoảng trắng thừa trong dòng; thay control character lỗi bằng dấu cách để không dính từ.
- Dấu tiếng Việt, chữ hoa/thường, dấu câu, số điều/khoản/điểm và nội dung pháp lý lặp lại: giữ nguyên.
- ID và qrels: không chuẩn hóa; qrels được sao chép nguyên trạng.

Dữ liệu trong `data/raw` không bị thay đổi. Ký tự lỗi thay bằng khoảng trắng, không tự đoán phần nội dung đã mất.

In [7]:
import re
import unicodedata

NORMALIZATION_MARKER = '__text_was_normalized__'


def normalize_legal_text(text):
    if text is None:
        return None
    if not isinstance(text, str):
        raise TypeError(f'Expected text to be str or None, got {type(text).__name__}')

    text = unicodedata.normalize('NFC', text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    for character in ('\ufeff', '\u200b', '\u2060'):
        text = text.replace(character, '')
    text = text.replace('\ufffd', ' ')
    text = ''.join(
        ' ' if unicodedata.category(character) == 'Cc' and character not in '\n\t' else character
        for character in text
    )

    normalized_lines = [re.sub(r'[^\S\n]+', ' ', line).strip() for line in text.split('\n')]
    output_lines = []
    for line in normalized_lines:
        if line:
            output_lines.append(line)
        elif output_lines and output_lines[-1] != '':
            output_lines.append('')

    while output_lines and output_lines[-1] == '':
        output_lines.pop()
    return '\n'.join(output_lines)


def normalize_nested_text(value):
    if isinstance(value, str):
        return normalize_legal_text(value)
    if isinstance(value, dict):
        return {key: normalize_nested_text(nested) for key, nested in value.items()}
    if isinstance(value, list):
        return [normalize_nested_text(nested) for nested in value]
    if isinstance(value, tuple):
        return tuple(normalize_nested_text(nested) for nested in value)
    return value


def normalize_dataset(dataset, text_columns, description):
    missing_columns = set(text_columns) - set(dataset.column_names)
    if missing_columns:
        raise ValueError(f'{description}: missing text columns {sorted(missing_columns)}')
    if NORMALIZATION_MARKER in dataset.column_names:
        raise ValueError(f'{description}: reserved column already exists: {NORMALIZATION_MARKER}')

    def normalize_row(row):
        normalized = {column: normalize_nested_text(row[column]) for column in text_columns}
        normalized[NORMALIZATION_MARKER] = any(
            normalized[column] != row[column] for column in text_columns
        )
        return normalized

    normalized_dataset = dataset.map(normalize_row, desc=description)
    changed_rows = sum(normalized_dataset[NORMALIZATION_MARKER])
    return normalized_dataset.remove_columns(NORMALIZATION_MARKER), changed_rows


def _get_columns(dataset):
    if hasattr(dataset, 'column_names'):
        return list(dataset.column_names)
    return list(dataset.columns)


def _iter_records(dataset):
    if hasattr(dataset, 'iterrows'):
        for _, row in dataset.iterrows():
            yield row.to_dict()
    else:
        yield from dataset


def _is_missing_scalar(value):
    if value is None or type(value).__name__ == 'NAType':
        return True
    return isinstance(value, float) and value != value


def _same_value(left, right):
    if left is right:
        return True
    if _is_missing_scalar(left) or _is_missing_scalar(right):
        return _is_missing_scalar(left) and _is_missing_scalar(right)
    if isinstance(left, dict) and isinstance(right, dict):
        return left.keys() == right.keys() and all(_same_value(left[key], right[key]) for key in left)
    if isinstance(left, (list, tuple)) and isinstance(right, type(left)):
        return len(left) == len(right) and all(_same_value(a, b) for a, b in zip(left, right))
    try:
        result = left == right
        return bool(result)
    except (TypeError, ValueError):
        return False


def _content_signature(value):
    if isinstance(value, str):
        text = unicodedata.normalize('NFC', value)
        text = text.replace('\r\n', '\n').replace('\r', '\n')
        ignored = {'\ufeff', '\u200b', '\u2060', '\ufffd'}
        return ''.join(
            character for character in text
            if character not in ignored
            and not character.isspace()
            and unicodedata.category(character) != 'Cc'
        )
    if isinstance(value, dict):
        return {key: _content_signature(nested) for key, nested in value.items()}
    if isinstance(value, list):
        return [_content_signature(nested) for nested in value]
    if isinstance(value, tuple):
        return tuple(_content_signature(nested) for nested in value)
    return value


def validate_cleaned_dataset(raw_dataset, cleaned_dataset, text_columns, name, id_column='id'):
    raw_columns = _get_columns(raw_dataset)
    cleaned_columns = _get_columns(cleaned_dataset)
    if raw_columns != cleaned_columns:
        raise AssertionError(f'{name}: columns changed: raw={raw_columns}, cleaned={cleaned_columns}')
    missing_columns = set(text_columns) - set(raw_columns)
    if missing_columns:
        raise ValueError(f'{name}: missing text columns {sorted(missing_columns)}')
    if id_column in raw_columns and id_column not in cleaned_columns:
        raise AssertionError(f'{name}: ID column {id_column!r} was removed')
    if len(raw_dataset) != len(cleaned_dataset):
        raise AssertionError(f'{name}: row count changed from {len(raw_dataset)} to {len(cleaned_dataset)}')

    changed_rows = 0
    for row_index, (raw_row, cleaned_row) in enumerate(zip(
        _iter_records(raw_dataset), _iter_records(cleaned_dataset)
    )):
        if id_column in raw_columns and not _same_value(raw_row[id_column], cleaned_row[id_column]):
            raise AssertionError(f'{name}: ID changed at row {row_index}')
        for column in raw_columns:
            if column not in text_columns:
                if not _same_value(raw_row[column], cleaned_row[column]):
                    raise AssertionError(f'{name}: non-text field {column!r} changed at row {row_index}')
                continue
            raw_value = raw_row[column]
            cleaned_value = cleaned_row[column]
            if not _same_value(normalize_nested_text(raw_value), cleaned_value):
                raise AssertionError(f'{name}: unexpected text change in {column!r} at row {row_index}')
            if not _same_value(_content_signature(raw_value), _content_signature(cleaned_value)):
                raise AssertionError(f'{name}: legal content lost from {column!r} at row {row_index}')
            if not _same_value(raw_value, cleaned_value):
                changed_rows += 1
    return {'dataset': name, 'rows': len(raw_dataset), 'text_fields_changed': changed_rows}


def validate_ground_truth_after_cleaning(raw_qrels, cleaned_qrels, raw_queries, cleaned_queries,
                                         raw_corpus, cleaned_corpus):
    for label, raw_dataset, cleaned_dataset in (
        ('queries', raw_queries, cleaned_queries),
        ('corpus', raw_corpus, cleaned_corpus),
        ('qrels', raw_qrels, cleaned_qrels),
    ):
        if len(raw_dataset) != len(cleaned_dataset):
            raise AssertionError(f'{label}: row count changed during cleaning')

    required_qrels_columns = {'query-id', 'corpus-id'}
    if not required_qrels_columns.issubset(_get_columns(raw_qrels)):
        raise ValueError(f'Qrels missing required columns: {sorted(required_qrels_columns - set(_get_columns(raw_qrels)))}')
    if _get_columns(raw_qrels) != _get_columns(cleaned_qrels):
        raise AssertionError('Qrels schema changed during cleaning')

    raw_query_ids = {row['id'] for row in _iter_records(raw_queries)}
    cleaned_query_ids = {row['id'] for row in _iter_records(cleaned_queries)}
    raw_corpus_ids = {row['id'] for row in _iter_records(raw_corpus)}
    cleaned_corpus_ids = {row['id'] for row in _iter_records(cleaned_corpus)}
    if raw_query_ids != cleaned_query_ids:
        raise AssertionError('Query IDs changed during cleaning')
    if raw_corpus_ids != cleaned_corpus_ids:
        raise AssertionError('Corpus IDs changed during cleaning')

    raw_qrel_rows = list(_iter_records(raw_qrels))
    cleaned_qrel_rows = list(_iter_records(cleaned_qrels))
    if not all(_same_value(raw, cleaned) for raw, cleaned in zip(raw_qrel_rows, cleaned_qrel_rows)):
        raise AssertionError('Qrels pairs or relevance scores changed during cleaning')

    query_column, corpus_column = 'query-id', 'corpus-id'
    missing_query_links = sum(
        row[query_column] not in cleaned_query_ids for row in cleaned_qrel_rows
    )
    missing_corpus_links = sum(
        row[corpus_column] not in cleaned_corpus_ids for row in cleaned_qrel_rows
    )
    if missing_query_links or missing_corpus_links:
        raise AssertionError(
            f'Ground-truth links broken after cleaning: {missing_query_links} query IDs, '
            f'{missing_corpus_links} corpus IDs'
        )
    return {
        'queries': len(cleaned_query_ids),
        'corpus_documents': len(cleaned_corpus_ids),
        'qrels': len(cleaned_qrel_rows),
        'broken_query_links': missing_query_links,
        'broken_corpus_links': missing_corpus_links,
    }


print('Conservative legal-text normalization helpers ready.')

Conservative legal-text normalization helpers ready.


### [TÁI SỬ DỤNG] Import thư viện dùng trong notebook

In [8]:
import pandas as pd
from datasets import DatasetDict, load_from_disk

## 3. Bộ dữ liệu ViQuAD 2.0

### [TÁI SỬ DỤNG] Đọc ViQuAD đã lưu trên Google Drive

In [9]:
if not VIQUAD_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy dữ liệu ViQuAD đã lưu: {VIQUAD_DIR}')

viquad = load_from_disk(str(VIQUAD_DIR))
print(viquad)

DatasetDict({
    train: Dataset({
        features: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers'],
        num_rows: 28454
    })
    validation: Dataset({
        features: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers'],
        num_rows: 3814
    })
    test: Dataset({
        features: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers'],
        num_rows: 7301
    })
})


### [TÁI SỬ DỤNG] Kiểm tra cấu trúc và mẫu ViQuAD

In [10]:
for split_name, split_data in viquad.items():
    print(f'\n--- {split_name} ---')
    print('Số mẫu:', len(split_data))
    print('Các cột:', split_data.column_names)
    print('Features:', split_data.features)
    if len(split_data) > 0:
        print('Mẫu đầu tiên:', split_data[0])


--- train ---
Số mẫu: 28454
Các cột: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers']
Features: {'id': Value('string'), 'uit_id': Value('string'), 'title': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answers': {'text': List(Value('string')), 'answer_start': List(Value('int32'))}, 'is_impossible': Value('bool'), 'plausible_answers': {'text': List(Value('string')), 'answer_start': List(Value('int32'))}}
Mẫu đầu tiên: {'id': '0001-0001-0001', 'uit_id': 'uit_000001', 'title': 'Phạm Văn Đồng', 'context': 'Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch H

## 4. Bộ dữ liệu Zalo Legal

### [TÁI SỬ DỤNG] Đọc corpus, queries và qrels đã lưu trên Google Drive

Ba phần này tạo thành bộ dữ liệu retrieval: corpus là tài liệu, queries là câu hỏi và qrels là nhãn liên kết câu hỏi với tài liệu.

In [11]:
if not all((ZALO_DIR / name).is_dir() for name in ('corpus', 'queries', 'qrels')):
    raise FileNotFoundError(f'Chưa tìm thấy đủ corpus, queries, qrels tại {ZALO_DIR}')

corpus = load_from_disk(str(ZALO_DIR / 'corpus'))
queries = load_from_disk(str(ZALO_DIR / 'queries'))
qrels = load_from_disk(str(ZALO_DIR / 'qrels'))

print('Corpus:', len(corpus))
print('Queries:', len(queries))
print('Qrels:', len(qrels))

Corpus: 61425
Queries: 818
Qrels: 793


### [TÁI SỬ DỤNG] Xem mẫu Zalo Legal

In [12]:
print('Corpus sample:', corpus[0])
print('\nQuery sample:', queries[0])
print('\nQrels sample:', qrels[0])

Corpus sample: {'id': '01/2009/tt-bnn+1', 'title': 'Điều 1. Phạm vi áp dụng', 'text': 'Thông tư này hướng dẫn tuần tra, canh gác bảo vệ đê Điều trong mùa lũ đối với các tuyến đê sông được phân loại, phân cấp theo quy định tại Điều 4 của Luật Đê Điều.'}

Query sample: {'id': '268afbc1f6756c6ef1f37f66f5fb85bd', 'text': 'Mức phạt khi quay đầu xe ô tô trên đường cao tốc'}

Qrels sample: {'query-id': 'a7365b07d9b07a2e38046da5ebd2070b', 'corpus-id': '100/2019/nđ-cp+6', 'score': 1}


### [TÁI SỬ DỤNG] Kiểm tra schema và tính hợp lệ của ID

In [13]:
print('Corpus columns:', corpus.column_names)
print('Queries columns:', queries.column_names)
print('Qrels columns:', qrels.column_names)

corpus_ids = set(corpus['id'])
query_ids = set(queries['id'])
qrels_query_ids = set(qrels['query-id'])
qrels_corpus_ids = set(qrels['corpus-id'])

print('\nTất cả query ID trong qrels đều tồn tại:', qrels_query_ids.issubset(query_ids))
print('Tất cả corpus ID trong qrels đều tồn tại:', qrels_corpus_ids.issubset(corpus_ids))
print('Số query có nhãn:', len(qrels_query_ids))
print('Số corpus được gán nhãn:', len(qrels_corpus_ids))

Corpus columns: ['id', 'title', 'text']
Queries columns: ['id', 'text']
Qrels columns: ['query-id', 'corpus-id', 'score']

Tất cả query ID trong qrels đều tồn tại: True
Tất cả corpus ID trong qrels đều tồn tại: True
Số query có nhãn: 788
Số corpus được gán nhãn: 447


## 5. Data Quality Inspection

Kiểm tra giá trị thiếu/trống, ID thiếu/trùng, bản ghi trùng hoàn toàn, thống kê độ dài văn bản và ground truth của Zalo Legal. Điểm qrels lớn hơn 0 được xem là nhãn liên quan. Các hàm có thể tái sử dụng cho split khác.

In [14]:
import hashlib
import json
import math
from collections.abc import Mapping
from statistics import mean, median


def _is_missing(value):
    return value is None or (isinstance(value, float) and math.isnan(value))


def _flatten_text(value):
    if isinstance(value, str):
        yield value
    elif isinstance(value, Mapping):
        for nested_value in value.values():
            yield from _flatten_text(nested_value)
    elif isinstance(value, (list, tuple)):
        for nested_value in value:
            yield from _flatten_text(nested_value)


def _length_summary(lengths):
    if not lengths:
        return {'count': 0, 'min': None, 'mean': None, 'median': None, 'p95': None, 'max': None}
    ordered = sorted(lengths)
    p95_index = max(0, math.ceil(0.95 * len(ordered)) - 1)
    return {
        'count': len(ordered),
        'min': ordered[0],
        'mean': round(mean(ordered), 2),
        'median': median(ordered),
        'p95': ordered[p95_index],
        'max': ordered[-1],
    }


def inspect_records(dataset, name, text_columns=None, id_column='id'):
    columns = list(dataset.column_names)
    if id_column not in columns:
        raise ValueError(f'{name}: không tìm thấy cột ID {id_column!r}; columns={columns}')

    if text_columns is None:
        preferred = ('text', 'context', 'question', 'title', 'content', 'paragraph', 'answers')
        text_columns = [column for column in preferred if column in columns]
        if not text_columns and len(dataset):
            sample = dataset[0]
            text_columns = [
                column for column in columns
                if column != id_column and any(_flatten_text(sample.get(column)))
            ]
    unknown_columns = set(text_columns) - set(columns)
    if unknown_columns:
        raise ValueError(f'{name}: không tìm thấy cột văn bản {sorted(unknown_columns)}')

    seen_ids = set()
    seen_records = set()
    missing_ids = 0
    duplicate_ids = 0
    duplicate_records = 0
    missing_by_column = {column: 0 for column in columns}
    empty_text_by_column = {column: 0 for column in text_columns}
    lengths_by_column = {column: [] for column in text_columns}
    row_count = 0

    for row in dataset:
        row_count += 1
        fingerprint = hashlib.sha256(
            json.dumps(row, sort_keys=True, ensure_ascii=False, default=str).encode('utf-8')
        ).digest()
        if fingerprint in seen_records:
            duplicate_records += 1
        seen_records.add(fingerprint)

        for column in columns:
            if _is_missing(row.get(column)):
                missing_by_column[column] += 1

        raw_id = row.get(id_column)
        normalized_id = None if _is_missing(raw_id) else str(raw_id).strip()
        if not normalized_id:
            missing_ids += 1
        elif normalized_id in seen_ids:
            duplicate_ids += 1
        else:
            seen_ids.add(normalized_id)

        for column in text_columns:
            parts = list(_flatten_text(row.get(column)))
            text = ' '.join(part.strip() for part in parts if part.strip())
            if not text:
                empty_text_by_column[column] += 1
            else:
                lengths_by_column[column].append(len(text))

    return {
        'dataset': name,
        'rows': row_count,
        'missing_ids': missing_ids,
        'duplicate_ids': duplicate_ids,
        'duplicate_records': duplicate_records,
        'missing_by_column': missing_by_column,
        'empty_text_by_column': empty_text_by_column,
        'text_lengths': {column: _length_summary(lengths) for column, lengths in lengths_by_column.items()},
    }


def inspect_ground_truth(qrels, queries, corpus, score_column='score'):
    required = {'query-id', 'corpus-id'}
    columns = set(qrels.column_names)
    if not required.issubset(columns):
        raise ValueError(f'Qrels thiếu cột bắt buộc: {sorted(required - columns)}')

    def ids_and_missing(dataset):
        ids = set()
        missing = 0
        for row in dataset:
            value = row.get('id')
            normalized = None if _is_missing(value) else str(value).strip()
            if normalized:
                ids.add(normalized)
            else:
                missing += 1
        return ids, missing

    query_ids, missing_query_ids = ids_and_missing(queries)
    corpus_ids, missing_corpus_ids = ids_and_missing(corpus)
    seen_pairs = set()
    duplicate_pairs = 0
    missing_qrel_query_ids = 0
    missing_qrel_corpus_ids = 0
    unknown_query_ids = set()
    unknown_corpus_ids = set()
    positive_query_ids = set()
    missing_scores = 0
    invalid_scores = 0
    positive_qrels = 0
    non_positive_qrels = 0

    for row in qrels:
        raw_query_id = row.get('query-id')
        raw_corpus_id = row.get('corpus-id')
        query_id = None if _is_missing(raw_query_id) else str(raw_query_id).strip()
        corpus_id = None if _is_missing(raw_corpus_id) else str(raw_corpus_id).strip()
        if not query_id:
            missing_qrel_query_ids += 1
        elif query_id not in query_ids:
            unknown_query_ids.add(query_id)
        if not corpus_id:
            missing_qrel_corpus_ids += 1
        elif corpus_id not in corpus_ids:
            unknown_corpus_ids.add(corpus_id)

        if query_id and corpus_id:
            pair = (query_id, corpus_id)
            if pair in seen_pairs:
                duplicate_pairs += 1
            seen_pairs.add(pair)

        if score_column in columns:
            score = row.get(score_column)
            if _is_missing(score):
                missing_scores += 1
            else:
                try:
                    numeric_score = float(score)
                except (TypeError, ValueError):
                    invalid_scores += 1
                else:
                    if numeric_score > 0:
                        positive_qrels += 1
                        if query_id and query_id in query_ids:
                            positive_query_ids.add(query_id)
                    else:
                        non_positive_qrels += 1

    return {
        'queries': len(query_ids),
        'corpus_documents': len(corpus_ids),
        'qrels': len(qrels),
        'missing_query_ids_in_source': missing_query_ids,
        'missing_corpus_ids_in_source': missing_corpus_ids,
        'missing_query_ids_in_qrels': missing_qrel_query_ids,
        'missing_corpus_ids_in_qrels': missing_qrel_corpus_ids,
        'unknown_query_ids': len(unknown_query_ids),
        'unknown_corpus_ids': len(unknown_corpus_ids),
        'duplicate_query_corpus_pairs': duplicate_pairs,
        'queries_without_positive_ground_truth': len(query_ids - positive_query_ids) if score_column in columns else None,
        'positive_qrels': positive_qrels if score_column in columns else None,
        'non_positive_qrels': non_positive_qrels if score_column in columns else None,
        'missing_scores': missing_scores if score_column in columns else None,
        'invalid_scores': invalid_scores if score_column in columns else None,
    }


print('Data quality helpers ready.')

Data quality helpers ready.


In [15]:
record_reports = []
for split_name, split_data in viquad.items():
    record_reports.append(inspect_records(split_data, f'ViQuAD/{split_name}'))

record_reports.extend([
    inspect_records(corpus, 'Zalo/corpus'),
    inspect_records(queries, 'Zalo/queries'),
])

summary_rows = [
    {
        'dataset': report['dataset'],
        'rows': report['rows'],
        'missing_ids': report['missing_ids'],
        'duplicate_ids': report['duplicate_ids'],
        'duplicate_records': report['duplicate_records'],
        'missing_values': sum(report['missing_by_column'].values()),
    }
    for report in record_reports
]
display(pd.DataFrame(summary_rows).set_index('dataset'))

missing_rows = [
    {'dataset': report['dataset'], 'column': column, 'missing': count}
    for report in record_reports
    for column, count in report['missing_by_column'].items()
    if count
]
empty_rows = [
    {'dataset': report['dataset'], 'column': column, 'empty': count}
    for report in record_reports
    for column, count in report['empty_text_by_column'].items()
    if count
]
length_rows = [
    {'dataset': report['dataset'], 'column': column, **stats}
    for report in record_reports
    for column, stats in report['text_lengths'].items()
]

print('Thiếu giá trị theo cột (chỉ hiện cột có lỗi):')
display(pd.DataFrame(missing_rows) if missing_rows else pd.DataFrame(columns=['dataset', 'column', 'missing']))
print('Văn bản rỗng theo cột:')
display(pd.DataFrame(empty_rows) if empty_rows else pd.DataFrame(columns=['dataset', 'column', 'empty']))
print('Độ dài văn bản (đơn vị: ký tự):')
display(pd.DataFrame(length_rows).set_index(['dataset', 'column']))

ground_truth_report = inspect_ground_truth(qrels, queries, corpus)
print('Ground truth / qrels quality:')
display(pd.DataFrame([ground_truth_report]).T.rename(columns={0: 'value'}))

,rows,missing_ids,duplicate_ids,duplicate_records,missing_values
dataset,,,,,
ViQuAD/train,28454,0,0,0,19238
ViQuAD/validation,3814,0,0,0,2653
ViQuAD/test,7301,0,0,0,14602
Zalo/corpus,61425,0,0,0,0
Zalo/queries,818,0,30,30,0


Thiếu giá trị theo cột (chỉ hiện cột có lỗi):


,dataset,column,missing
0,ViQuAD/train,plausible_answers,19238
1,ViQuAD/validation,plausible_answers,2653
2,ViQuAD/test,answers,7301
3,ViQuAD/test,plausible_answers,7301


Văn bản rỗng theo cột:


,dataset,column,empty
0,ViQuAD/train,answers,9216
1,ViQuAD/validation,answers,1161
2,ViQuAD/test,answers,7301
3,Zalo/corpus,text,357


Độ dài văn bản (đơn vị: ký tự):


count    min     mean  median     p95       max
dataset           column                                                   
ViQuAD/train      context   28454  465.0   833.34   743.0  1460.0    7221.0
                  question  28454    3.0    65.21    62.0   106.0     232.0
                  title     28454    2.0    13.88    13.0    28.0      45.0
                  answers   19238    1.0    45.19    27.0   141.0     740.0
ViQuAD/validation context    3814  486.0   797.20   721.0  1322.0    2852.0
                  question   3814   10.0    64.56    62.0   105.0     245.0
                  title      3814    4.0    12.81    11.0    21.0      24.0
                  answers    2653    1.0    78.21    42.0   264.0     824.0
ViQuAD/test       context    7301  474.0   821.35   749.0  1384.0    3760.0
                  question   7301   15.0    64.98    63.0   102.0     168.0
                  title      7301    3.0    11.79    11.0    21.0      28.0
                  answers       0    NaN      NaN     NaN     NaN       NaN
Zalo/corpus       text      61068    2.0  1306.03   834.0  3672.0  252881.0
                  title     61425    6.0    59.36    49.0   131.0    1717.0
Zalo/queries      text        818   16.0    83.78    80.0   131.0     164.0

Ground truth / qrels quality:


,value
queries,788
corpus_documents,61425
qrels,793
missing_query_ids_in_source,0
missing_corpus_ids_in_source,0
missing_query_ids_in_qrels,0
missing_corpus_ids_in_qrels,0
unknown_query_ids,0
unknown_corpus_ids,0
duplicate_query_corpus_pairs,0


## 6. Text Cleaning & Normalization

### [CHẠY MỘT LẦN / KHI CẬP NHẬT] Chuẩn hóa trong bộ nhớ

Chạy các cell theo thứ tự từ trên xuống. Trước cell này, cần chạy cell **[TÁI SỬ DỤNG] Hàm chuẩn hóa văn bản pháp lý** để định nghĩa `normalize_dataset`. Cell này chỉ tạo dữ liệu cleaned trong bộ nhớ, không ghi lên Drive. Sau đó chạy cell kiểm định; lưu processed là bước riêng ở cuối notebook.

In [16]:
required_helpers = ('normalize_dataset', 'validate_cleaned_dataset', 'validate_ground_truth_after_cleaning')
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        'Chưa định nghĩa hàm chuẩn hóa/kiểm định: '
        f'{missing_helpers}. Hãy chạy cell [TÁI SỬ DỤNG] Hàm chuẩn hóa văn bản pháp lý '
        'ở mục 2 trước, hoặc chọn Runtime > Run all.'
    )

required_datasets = ('viquad', 'corpus', 'queries', 'qrels')
missing_datasets = [name for name in required_datasets if name not in globals()]
if missing_datasets:
    raise RuntimeError(
        f'Dữ liệu chưa được nạp: {missing_datasets}. Hãy chạy các cell tải dữ liệu từ Drive trước.'
    )

viquad_raw = viquad
corpus_raw = corpus
queries_raw = queries
qrels_raw = qrels

normalized_viquad = {}
normalization_summary = []
for split_name, split_data in viquad_raw.items():
    normalized_split, changed_rows = normalize_dataset(
        split_data,
        ['title', 'context', 'question', 'answers', 'plausible_answers'],
        f'Normalize ViQuAD/{split_name}',
    )
    normalized_viquad[split_name] = normalized_split
    normalization_summary.append({
        'dataset': f'ViQuAD/{split_name}',
        'rows': len(split_data),
        'rows_with_text_changes': changed_rows,
    })

normalized_corpus, changed_corpus_rows = normalize_dataset(
    corpus_raw, ['title', 'text'], 'Normalize Zalo corpus'
)
normalized_queries, changed_query_rows = normalize_dataset(
    queries_raw, ['text'], 'Normalize Zalo queries'
)
qrels_clean = qrels_raw

normalization_summary.extend([
    {'dataset': 'Zalo/corpus', 'rows': len(corpus_raw), 'rows_with_text_changes': changed_corpus_rows},
    {'dataset': 'Zalo/queries', 'rows': len(queries_raw), 'rows_with_text_changes': changed_query_rows},
])

display(pd.DataFrame(normalization_summary).set_index('dataset'))
print('Cleaned ViQuAD, corpus, and queries are ready in memory; qrels are unchanged.')

,rows,rows_with_text_changes
dataset,,
ViQuAD/train,28454,107
ViQuAD/validation,3814,7
ViQuAD/test,7301,74
Zalo/corpus,61425,2344
Zalo/queries,818,4


Cleaned ViQuAD, corpus, and queries are ready in memory; qrels are unchanged.


## 7. Automated Post-clean Validation

### [TÁI SỬ DỤNG] Kiểm tra bảo toàn nội dung và ground truth

Cell này không ghi dữ liệu lên Drive. Nó dừng bằng AssertionError nếu mất/thay đổi ký tự ngoài các chuẩn hóa cho phép, ID/số dòng/schema thay đổi, qrels bị sửa hoặc liên kết query-document bị hỏng. Các biến `df_corpus_clean`, `df_queries_clean`, `df_qrels` tương ứng với dữ liệu Zalo đã làm sạch và qrels nguyên trạng.

In [17]:
df_corpus_clean = normalized_corpus
df_queries_clean = normalized_queries
df_qrels = qrels_clean

preservation_reports = []
for split_name, raw_split in viquad_raw.items():
    preservation_reports.append(validate_cleaned_dataset(
        raw_split,
        normalized_viquad[split_name],
        ['title', 'context', 'question', 'answers', 'plausible_answers'],
        f'ViQuAD/{split_name}',
    ))

preservation_reports.extend([
    validate_cleaned_dataset(
        corpus_raw, df_corpus_clean, ['title', 'text'], 'Zalo/corpus'
    ),
    validate_cleaned_dataset(
        queries_raw, df_queries_clean, ['text'], 'Zalo/queries'
    ),
])

ground_truth_validation = validate_ground_truth_after_cleaning(
    qrels_raw,
    df_qrels,
    queries_raw,
    df_queries_clean,
    corpus_raw,
    df_corpus_clean,
)
display(pd.DataFrame(preservation_reports).set_index('dataset'))
display(pd.DataFrame([ground_truth_validation]).T.rename(columns={0: 'value'}))
print('PASS: cleaned text preserved; IDs, records, qrels, scores, and all ground-truth links are intact.')

,rows,text_fields_changed
dataset,,
ViQuAD/train,28454,108
ViQuAD/validation,3814,7
ViQuAD/test,7301,75
Zalo/corpus,61425,2416
Zalo/queries,818,4


,value
queries,788
corpus_documents,61425
qrels,793
broken_query_links,0
broken_corpus_links,0


PASS: cleaned text preserved; IDs, records, qrels, scores, and all ground-truth links are intact.


## 8. Lưu processed lên Google Drive (tùy chọn)

Đây là cell ghi dữ liệu riêng, không nằm trong block cleaning hoặc validation. Chỉ chạy sau khi cell kiểm định đạt. Cell sẽ từ chối ghi đè thư mục processed đã có dữ liệu.

In [18]:
PROCESSED_VIQUAD_DIR = PROCESSED_DIR / 'viquad'
PROCESSED_ZALO_DIR = PROCESSED_DIR / 'zalo_legal'

for output_dir in (PROCESSED_VIQUAD_DIR, PROCESSED_ZALO_DIR):
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(
            f'{output_dir} đã có dữ liệu. Để tránh ghi đè, hãy chọn thư mục output mới '
            'hoặc chủ động dọn bản processed cũ trước khi chạy lại.'
        )

DatasetDict(normalized_viquad).save_to_disk(str(PROCESSED_VIQUAD_DIR))
df_corpus_clean.save_to_disk(str(PROCESSED_ZALO_DIR / 'corpus'))
df_queries_clean.save_to_disk(str(PROCESSED_ZALO_DIR / 'queries'))
df_qrels.save_to_disk(str(PROCESSED_ZALO_DIR / 'qrels'))

print('Processed ViQuAD:', PROCESSED_VIQUAD_DIR)
print('Processed Zalo Legal:', PROCESSED_ZALO_DIR)

Saving the dataset (0/1 shards):   0%|          | 0/28454 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3814 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7301 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/61425 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/793 [00:00<?, ? examples/s]

Processed ViQuAD: /content/drive/MyDrive/Vietnamese-RAG-Research/data/processed/viquad
Processed Zalo Legal: /content/drive/MyDrive/Vietnamese-RAG-Research/data/processed/zalo_legal
